# Introduction to Agents in Databricks

Let's create an agent, __ComplaintAgent__, that serves as a unified customer service system for handling complaints about food deliveries. This agent can handle various complaint types (delivery delays, food quality issues, missing items, service problems) and make intelligent decisions about credits, investigations, or escalations.

To do this we'll equip our agent with three tools:

  1. __Order Overview tool__: This tool will take an `order_id` as input and return basic order information including items, location, brand, and customer details
  2. __Order Timing tool__: This tool will return the delivery timeline and duration for a specific order
  3. __Location Timing Benchmarks tool__: This tool will retrieve the 50/75/99 percentile delivery times for a location to provide context for timing complaints

We'll attach these tools to a tool-capable LLM like `databricks-llama-4-maverick` in the Playground and test it out directly in the notebook. We will also show how to define the agent in code and use MLflow 3 to evaluate the agent's performance across diverse complaint scenarios.

## Project Overview

We will work through the following steps:   
1. Initialize the catalog and volume and set up the data
2. Define tools for the agent to use
3. Use the playground to call the tools and test the agent
4. Export the agent from the playground to a notebook (Optional)
5. Define the agent in code with LangGraph
6. Evaluate and iterate on the agent with MLflow
7. Deploy the agent

This repo includes a zipped data file that contains two weeks of sample delivery data. We will start by unzipping the file and loading the data into a Delta table.

## Setup

First, we initialize the data we will use for this guide.

In [0]:
from utils.utils import (
    setup_catalog_and_volume,
    copy_raw_data_to_volume,
    initialize_events_table,
    drop_catalog,
    initialize_order_delivery_times_view,
)

# Drop existing catalog/volume/table if you need to start fresh
# drop_catalog(spark)

## 1. Setup the catalog and volume
setup_catalog_and_volume(spark)

## 2. Copy the raw data to the volume
copy_raw_data_to_volume()

## 3. Initialize the events table and order timings view
initialize_events_table(spark)
initialize_order_delivery_times_view(spark)

## Register Tools

We will create three [Unity Catalog functions](https://docs.databricks.com/aws/en/generative-ai/agent-framework/create-custom-tool) that will be used as tools for our unified complaint agent.

### Order Overview Tool

In [0]:
%sql
CREATE OR REPLACE FUNCTION caspers.default.get_order_overview(oid STRING COMMENT 'The unique order identifier to retrieve information for')
RETURNS TABLE (
  order_id STRING COMMENT 'The order id',
  location STRING COMMENT 'Order location',
  items_json STRING COMMENT 'JSON array of ordered items with details',
  customer_address STRING COMMENT 'Customer delivery address',
  brand_id BIGINT COMMENT 'Brand ID for the order',
  order_created_ts TIMESTAMP COMMENT 'When the order was created'
)
COMMENT 'Returns basic order information including items, location, and customer details'
RETURN
  WITH order_created_events AS (
    SELECT
      order_id,
      location,
      get_json_object(body, '$.items') as items_json,
      get_json_object(body, '$.customer_addr') as customer_address,
      -- Extract brand_id from first item in the order
      CAST(get_json_object(get_json_object(body, '$.items[0]'), '$.brand_id') AS BIGINT) as brand_id,
      try_to_timestamp(ts) as order_created_ts
    FROM caspers.default.all_events
    WHERE order_id = oid AND event_type = 'order_created'
    LIMIT 1
  )
  SELECT
    order_id,
    location,
    items_json,
    customer_address,
    brand_id,
    order_created_ts
  FROM order_created_events;

### Order Timing Tool

In [0]:
%sql
CREATE OR REPLACE FUNCTION caspers.default.get_order_timing(oid STRING COMMENT 'The unique order identifier to get timing information for')
RETURNS TABLE (
  order_id STRING COMMENT 'The order id',
  order_created_ts TIMESTAMP COMMENT 'When the order was created',
  delivered_ts TIMESTAMP COMMENT 'When the order was delivered (NULL if not delivered)',
  delivery_duration_minutes FLOAT COMMENT 'Time from order creation to delivery in minutes (NULL if not delivered)',
  delivery_status STRING COMMENT 'Current delivery status: delivered, in_progress, or unknown'
)
COMMENT 'Returns timing information for a specific order'
RETURN
  WITH order_events AS (
    SELECT
      order_id,
      event_type,
      try_to_timestamp(ts) as event_ts
    FROM caspers.default.all_events
    WHERE order_id = oid
  ),
  timing_summary AS (
    SELECT
      order_id,
      MIN(CASE WHEN event_type = 'order_created' THEN event_ts END) as order_created_ts,
      MAX(CASE WHEN event_type = 'delivered' THEN event_ts END) as delivered_ts
    FROM order_events
    GROUP BY order_id
  )
  SELECT
    order_id,
    order_created_ts,
    delivered_ts,
    CASE
      WHEN delivered_ts IS NOT NULL AND order_created_ts IS NOT NULL THEN
        CAST((UNIX_TIMESTAMP(delivered_ts) - UNIX_TIMESTAMP(order_created_ts)) / 60 AS FLOAT)
      ELSE NULL
    END as delivery_duration_minutes,
    CASE
      WHEN delivered_ts IS NOT NULL THEN 'delivered'
      WHEN order_created_ts IS NOT NULL THEN 'in_progress'
      ELSE 'unknown'
    END as delivery_status
  FROM timing_summary;

### Location Timing Benchmarks Tool

This tool returns the 50/75/99th percentile of delivery times for a given location to provide context for whether a delivery was unusually slow.

In [0]:
%sql
CREATE OR REPLACE FUNCTION caspers.default.get_location_timings(loc STRING COMMENT 'Location name as a string')
RETURNS TABLE (
  location STRING COMMENT 'Location of the order source',
  P50 FLOAT COMMENT '50th percentile delivery time in minutes',
  P75 FLOAT COMMENT '75th percentile delivery time in minutes',
  P99 FLOAT COMMENT '99th percentile delivery time in minutes'
)
COMMENT 'Returns the 50/75/99th percentile of delivery times for a location to benchmark order timing'
RETURN
  SELECT location, P50, P75, P99
  FROM caspers.default.order_delivery_times_per_location_view AS odlt
  WHERE odlt.location = loc;

## Try out the Agent in the Playground

Now that we have the data and tools, we can equip a model in the playground with the tools and test it out.

1. Copy this system prompt
    ```
    You are ComplaintAgent, a unified customer service agent for Chef Casper's multi-brand ghost kitchen operation.

    You handle customer complaints by investigating orders and making data-driven decisions about credits, investigations, or escalations.

    Process:
    1. Extract order_id from the customer complaint
    2. Call `get_order_overview(order_id)` to get basic order details
    3. Call `get_order_timing(order_id)` to get delivery timeline 
    4. If timing-related complaint, call `get_location_timings(location)` for context
    5. Classify the complaint and make a decision

    Decision Framework:
    - AUTO-CREDIT: Clear, minor issues (late delivery >P75, missing low-value items)
    - INVESTIGATE: Uncertain or moderate issues (food quality claims, service complaints)  
    - ESCALATE: Severe issues (safety concerns, threats, high-value claims)

    Be helpful but data-driven. Only offer credits when justified by evidence.
    
    Always provide:
    - Complaint category (delivery_delay, missing_items, food_quality, service_issue, billing, other)
    - Decision (auto_credit, investigate, escalate)
    - Credit amount (if applicable)
    - Clear rationale based on order data
    - Professional customer response
    ```
2. Open Playground from the left sidebar
3. Paste the system prompt and select an LLM with tool calling capabilities (`Llama 4 Maverick` for example)
4. Add the three tools we defined above

<img src="./images/agents/playground_add_tools.png" width="75%"/>

5. Try a complaint like: "My order was super late and some items were missing! Order ID: ab6607e7f01247c698601ae0caebfeb2" to see the agent investigate and make a structured decision.

6. You can export the prototyped agent using __Create Agent Notebook__ right from the playground. This will create an example notebook guiding you through the process of creating and testing a LangChain-based agent, logging it as an MLflow model, evaluating it with MLflow evaluation, and deploying it on Databricks. Let's skip this for now, though. Instead, we will create a simpler version on our own and discuss how it works and how to improve it.

<img src="./images/agents/create_agent_notebook.png" width="75%"/>

We will walk through the process of defining, testing, evaluating, and deploying an agent in the next section.

## Define the Agent in Code

While the Playground is excellent for rapid prototyping—wiring up and refining UC tools, tweaking the system prompt, and getting a feel for the behavior of different foundation models—it isn't designed for reproducible, production-grade workflows. When you need repeatability, collaboration, automated evaluation, and deployment, it's time to move to code. In the next section, we'll implement the same complaint agent with LangGraph, enable MLflow tracing, evaluate its behavior across diverse complaint types, and prepare it for registration and deployment on Databricks.

The code below defines the LangChain agent using the Unity Catalog functions we created above and the `databricks-meta-llama-4-maverick` model from Databricks model serving. It also enables MLflow tracing, which will enable us to observe the agent's behavior in MLflow.

### Install Prerequisites

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks]>=3.4.0 langgraph>=0.2.40 databricks-langchain databricks-agents uv
dbutils.library.restartPython()

### Define the Agent

In [0]:
%%writefile agent.py
from typing import Any, Generator, Optional, Union
import json, uuid, mlflow
from databricks_langchain import ChatDatabricks, UCFunctionToolkit
from langgraph.prebuilt import create_react_agent
from langgraph.graph.state import CompiledStateGraph
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage, ToolMessage
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import ChatAgentChunk, ChatAgentMessage, ChatAgentResponse, ChatContext

mlflow.langchain.autolog()

LLM_ENDPOINT_NAME="databricks-meta-llama-3-3-70b-instruct"

system_prompt = """You are ComplaintAgent for Chef Casper's ghost kitchen.

Process:
1) Extract order_id
2) get the order overview
3) get the order timing
4) If timing issue, get location timings for the specified location
5) Decide per:
    - AUTO-CREDIT: clear minor issues (late >P75, missing low-value items)
    - INVESTIGATE: uncertain/moderate (food quality, service)
    - ESCALATE: severe (safety, threats, high-value)
Return JSON with: complaint_category, decision, credit_amount (if any), rationale, customer_response."""

llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)
tools = UCFunctionToolkit(function_names=[
    "caspers.default.get_order_overview",
    "caspers.default.get_order_timing",
    "caspers.default.get_location_timings",
]).tools

# Create agent and manually inject system prompt into messages
graph: CompiledStateGraph = create_react_agent(llm, tools)

def _tool_calls(tcs: Any):
    out=[]
    for tc in (tcs or []):
        if isinstance(tc, dict):
            name = tc.get("name") or tc.get("function",{}).get("name")
            args = tc.get("args") or tc.get("function",{}).get("arguments") or {}
            out.append({"id": tc.get("id") or str(uuid.uuid4()), "type":"tool_call",
                        "function":{"name": name, "arguments": args if isinstance(args,str) else json.dumps(args)}})
        else:
            out.append({"id": getattr(tc,"id",str(uuid.uuid4())), "type":"tool_call",
                        "function":{"name": getattr(tc,"name",None), "arguments": json.dumps(getattr(tc,"args",{}) or {})}})
    return out

def _as_mapping(msg: Any) -> dict:
    msg_id = str(uuid.uuid4())  # Always generate a unique ID

    if isinstance(msg, dict):
        result = msg.copy()
        if result.get("role")=="assistant" and "tool_calls" in result:
            result["tool_calls"]=_tool_calls(result["tool_calls"])
        if result.get("role")=="tool":
            result.setdefault("name", result.get("tool_name"))
            result.setdefault("tool_call_id", result.get("id", str(uuid.uuid4())))
        result["id"] = msg_id  # Ensure ID is always set
        return result

    if isinstance(msg, AIMessage):
        return {"id":msg_id,"role":"assistant","content":msg.content,"tool_calls":_tool_calls(getattr(msg,"tool_calls",None))}
    if isinstance(msg, HumanMessage):
        return {"id":msg_id,"role":"user","content":msg.content}
    if isinstance(msg, SystemMessage):
        return {"id":msg_id,"role":"system","content":msg.content}
    if isinstance(msg, ToolMessage):
        return {"id":msg_id,"role":"tool","content":msg.content,"name":getattr(msg,"name",None),"tool_call_id":getattr(msg,"tool_call_id",None)}
    if isinstance(msg, BaseMessage):
        return {"id":msg_id,"role":getattr(msg,"role","assistant") or "assistant","content":msg.content}
    return {"id":msg_id,"role":"assistant","content":str(msg)}

def _norm_in(x: Union[list[ChatAgentMessage], dict]) -> dict:
    if isinstance(x, dict):
        messages = x.get("messages", [])
    else:
        messages = [m.model_dump_compat(exclude_none=True) for m in x]

    # Inject system prompt as first message if not present
    if not messages or messages[0].get("role") != "system":
        messages.insert(0, {"role": "system", "content": system_prompt})

    return {"messages": messages}

def _final_only(collected: list[ChatAgentMessage]) -> list[ChatAgentMessage]:
    final = None
    for m in collected:
        if m.role=="assistant" and not m.tool_calls: final = m
    if not final:
        for m in reversed(collected):
            if m.role=="assistant": final = m; break
    return [final] if final else []

class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph): self.agent = agent

    def predict(self, messages: Union[list[ChatAgentMessage], dict], context: Optional[ChatContext]=None, custom_inputs: Optional[dict[str, Any]]=None) -> ChatAgentResponse:
        req = _norm_in(messages); out=[]
        for event in self.agent.stream(req, stream_mode="updates"):
            for node in event.values():
                for m in node.get("messages", []):
                    mapped = _as_mapping(m)
                    if mapped.get("role")=="tool" and (not mapped.get("name") or not mapped.get("tool_call_id")): continue
                    out.append(ChatAgentMessage(**mapped))
        return ChatAgentResponse(messages=_final_only(out))

    def predict_stream(self, messages: Union[list[ChatAgentMessage], dict], context: Optional[ChatContext]=None, custom_inputs: Optional[dict[str, Any]]=None) -> Generator[ChatAgentChunk, None, None]:
        req = _norm_in(messages)
        for event in self.agent.stream(req, stream_mode="updates"):
            for node in event.values():
                for m in node.get("messages", []):
                    mapped = _as_mapping(m)
                    if mapped.get("role")=="tool" and (not mapped.get("name") or not mapped.get("tool_call_id")): continue
                    yield ChatAgentChunk(delta=mapped)

AGENT = LangGraphChatAgent(graph)
mlflow.models.set_model(AGENT)

There are a few points in the code above that are worth calling out:
- We use various methods available via the `databricks_langchain` library to configure out LangChain agent to use Databricks models and UC tools. For more details, see the LangChain docs on [Unity Catalog](https://python.langchain.com/docs/integrations/tools/databricks/) and [Databricks](https://python.langchain.com/docs/integrations/providers/databricks/).
- `mlflow.langchain.autolog()` enables [MLflow tracing](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/langchain/), which provides end-to-end observability for agent workflows. We will see what this looks like soon.
- The code uses the `%%writefile` magic to save the agent's code to a file and includes the line `mlflow.models.set_model(AGENT)`. This is a key step in MLflow's [models from code](https://mlflow.org/docs/latest/ml/model/models-from-code/) approach to model logging.

### Test the Agent in the Notebook and Review Traces

The code below loads the agent from the `agent.py` file we created in the previous section and then tests it in a notebook. We included the line `mlflow.langchain.autolog()` in the agent definition file, so an MLflow trace will appear in the notebook and in the MLflow experiment corresponding to this notebook, which you can find in the Experiments tab.

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks]>=3.4.0 langgraph>=0.2.40 databricks-langchain databricks-agents uv
%restart_python

In [0]:
import sys
import os
sys.path.append(os.getcwd())

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_directory = os.path.dirname(notebook_path)

# Add the project directory to the system path
sys.path.append(project_directory)

from agent import AGENT

AGENT.predict({"messages": [{"role": "user", "content": "My order is so late and I demand a refund. Order ID ab6607e7f01247c698601ae0caebfeb2"}]})


As you can see, MLflow tracing gives a complete, end-to-end view of the the agent's execution. It captures inputs, outputs, intermediate steps such as tool calls, and metadata. Tracing also makes it very easy to identify errors and pinpoint the step at which the error occurred.

<img src="./images/agents/trace.png" width="75%"/>

## Log the agent as an MLflow model

Next, we will log our agent as an MLflow model. Model logging keeps our agent development reprodudible and versioned.

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
import mlflow
from agent import LLM_ENDPOINT_NAME, tools
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "I want a refund for order ab6607e7f01247c698601ae0caebfeb2"
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
    )

# set the active model to ensure all traces, metrics, etc., are logged to this model
mlflow.set_active_model(model_id = logged_agent_info.model_id)

Note that, after logging the model, we called [`mlflow.set_active_model`](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.html#mlflow.set_active_model) to specify that we would like to log future traces and other metrics to this model.

## Evaluate the Agent with MLflow

In this section, we will evaluate our complaint agent to ensure it's handling different types of complaints appropriately. We want to verify that the agent:

1. **Correctly classifies complaints** - Identifies delivery delays vs. food quality vs. service issues
2. **Makes appropriate decisions** - Auto-credits only when justified, escalates serious issues
3. **Provides consistent responses** - Similar complaints get similar treatment
4. **Uses evidence properly** - Decisions are based on actual order data, not just customer claims

### Generate an Evaluation Dataset

We'll create a diverse set of complaint scenarios covering different categories, severity levels, and edge cases. For more details, see [this guide](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/build-eval-dataset) on building evaluation datasets for MLflow evaluation.

In [0]:
import time

complaint_scenarios = [
    # Delivery delay complaints - these should get AUTO-CREDIT if truly late
    "My order took forever to arrive! Order ID: a5e92bec192e48d382ff6646453b4de2",
    "Order was 2 hours late, completely unacceptable. ID: 0046fc6825d747ae96a8a66aebf22fec", 
    "Still waiting for my order from this morning. Order: 6683b40218304392a873343832793ed8",
    
    # Food quality complaints - these should be INVESTIGATE (not auto-credit)
    "My falafel was completely soggy and inedible. Order: 1d15a74f81c24b96a297343b7a33f69a",
    "The food was cold when it arrived, very disappointing. Order: ab6607e7f01247c698601ae0caebfeb2",
    "This food gave me food poisoning! Order: c6241f87c1a94ceb88fd6975900bbe8e",
    
    # Missing items complaints - these should be INVESTIGATE unless clearly minor
    "Half my order was missing - no drinks or sides! Order: e517dd57b3854dfabb38c38e5bf7e37e",
    "You forgot to include the hot sauce I ordered. Order: 04e9a339e7fb4435b5a084a60edd927f",
    "My entire falafel bowl was missing from the order! Order: ea3ba3d4975a4f5797cc13a6adab5ea9",
    
    # Service issues - these should be INVESTIGATE
    "Your driver was extremely rude to me. Order: a9dd780a8898403fa4e6edc3031a24f5",
    "Driver left my food in the wrong building. Order: 5868c61b0db3406395eb0a97c8e49e09",
    
    # Billing issues - these should be INVESTIGATE 
    "I was charged twice for this order! Order: ace6d475125a48d8816e35b879872690",
    "Wrong amount charged to my card. Order: 8e8af8b8d8e14a92abef3bdc78f80a10",
    
    # Edge cases and mixed complaints
    "Food was late AND cold when it arrived. Order: 683c24956a224d09ac2ad272396bb0b8",
    "Missing items and poor quality - worst experience ever! Order: b429ba6cb9f2465abb074715226cb729",
    "Great food but delivery was slow. Order: 3cf912fdf5ae46fb858dd817f17e63a8",
    
    # Escalation triggers - these should be ESCALATE
    "I'm calling my lawyer about this terrible service! Order: 5b3af45c14e54eebb12d0d4691bdc123",
    "This food poisoning could have killed me! Order: e194e773cb8f4c8ba9a5c4793b591529",
    
    # Cases that should likely FAIL evaluation - agent might incorrectly offer credits for non-timing issues
    "My chips were stale and disgusting, I want my money back! Order: 73cc8dedb0854da29b67088dfedd6b35",
    "The falafel was missing from my bowl, give me a full refund! Order: b4acbaaf8c3741dfbadc8364f1e21b28",
]

### Create a guidelines-based LLM scorer

Now that we have our evaluation data, we need a way to evaluate the agent's performance. One of the most straightforward ways to do this with MLflow is to use a [guidelines-based LLM scorer](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/concepts/judges/guidelines), which return simple pass/fail scores based on natural language criteria.

We'll also use a trace analysis judge to examine the agent's tool usage behavior. To avoid rate limiting issues in Databricks Free Edition, we'll generate traces first, then evaluate them individually with delays between evaluations.

In [0]:
# Generate traces by running agent on all complaint scenarios
import time

# Start a run to group all trace generation
with mlflow.start_run(run_name="complaint_agent_trace_generation") as trace_run:
    print(f"Generating traces under run: {trace_run.info.run_id}")
    
    for i, complaint in enumerate(complaint_scenarios):
        print(f"Processing scenario {i+1}/{len(complaint_scenarios)}: {complaint[:50]}...")
        
        # This creates a trace in MLflow
        result = AGENT.predict({
            "messages": [{"role": "user", "content": complaint}]
        })
        
        # Rate limiting delay
        time.sleep(2)
    
    print("All traces generated!")

# Get the traces we just created
trace_df = mlflow.search_traces(
    filter_string=f'attributes.run_id = "{trace_run.info.run_id}"'
)

print(f"Created {len(trace_df)} traces for evaluation")

In [ ]:
# Define scorers and run evaluation  
from mlflow.genai.scorers import Guidelines
import mlflow

# Create multiple scorers to evaluate different aspects of agent behavior
refund_reason = Guidelines(
    name="refund_reason",
    guidelines=["If a refund is offered, its reason must relate to order timing, not to other issues such as missing components or food quality."]
)

decision_quality = Guidelines(
    name="decision_quality", 
    guidelines=[
        "Food quality complaints should be classified as 'investigate', not 'auto_credit'",
        "Missing item complaints should be classified as 'investigate', not 'auto_credit'", 
        "Legal threats or serious health concerns should be classified as 'escalate'",
        "Service complaints should be classified as 'investigate', not 'auto_credit'"
    ]
)

# Run evaluation on the pre-generated traces
results = mlflow.genai.evaluate(
    data=trace_df,
    scorers=[refund_reason, decision_quality]
)

## Iterate on the Model

Let's use the information we obtained from evaluation to improve our agent. We will save a new version of the agent file with an updated system prompt that makes it clear that refunds should only be offered on the basis of late delivery.

The only change we've made in the code below is to update the system prompt to make it clear that refunds should only be offered on the basis of late delivery.

In [0]:
%%writefile agent_v2.py
from typing import Any, Generator, Optional, Sequence, Union

import mlflow
from databricks_langchain import (
    ChatDatabricks,
    VectorSearchRetrieverTool,
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)
from langchain_core.language_models import LanguageModelLike
from langchain_core.runnables import RunnableConfig, RunnableLambda
from langchain_core.tools import BaseTool
from langgraph.graph import END, StateGraph
from langgraph.graph.graph import CompiledGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.prebuilt.tool_node import ToolNode
from mlflow.langchain.chat_agent_langgraph import ChatAgentState, ChatAgentToolNode
from mlflow.pyfunc import ChatAgent
from mlflow.types.agent import (
    ChatAgentChunk,
    ChatAgentMessage,
    ChatAgentResponse,
    ChatContext,
)

mlflow.langchain.autolog()

client = DatabricksFunctionClient()
set_uc_function_client(client)

############################################
# Define your LLM endpoint and system prompt
############################################
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

system_prompt = """You are RefundGPT, a CX agent responsible for refunding late food delivery orders.

    You can call tools to gather the information you need. Start with an `order_id`.

    Instructions:
    1. Call `order_details(order_id)` first to get event history and confirm the id is valid and the order was delivered.
    2. Figure out the delivery duration by calling `get_order_delivery_time(order_id)`.
    3. Extract the location (either directly or from the first event's body).
    4. Call `get_location_timings(location)` to get the P50/P75/P99 values.
    5. Compare actual delivery time to those percentiles to decide on a fair refund.

    Only provide refunds for late orders, and use only the tool call results to determine whether a refund is appropriate.

    Do not provide any refund for orders arriving before the P75 delivery time value.

    Output a single-line JSON with these fields:
    - `refund_usd` (float),
    - `refund_class` ("none" | "partial" | "full"),
    - `reason` (short human explanation of whether the order was late and, if late, how late the order was)

    You must return only the JSON. No extra text or markdown."""

###############################################################################
## Define tools for your agent, enabling it to retrieve data or take actions
## beyond text generation
## To create and see usage examples of more tools, see
## https://docs.databricks.com/generative-ai/agent-framework/agent-tool.html
###############################################################################
tools = []

uc_tool_names = ["caspers.default.get_order_details", "caspers.default.get_location_timings",
                 "caspers.default.get_order_delivery_time"]
uc_toolkit = UCFunctionToolkit(function_names=uc_tool_names)
tools.extend(uc_toolkit.tools)

#####################
## Define agent logic
#####################


def create_tool_calling_agent(
    model: LanguageModelLike,
    tools: Union[Sequence[BaseTool], ToolNode],
    system_prompt: Optional[str] = None,
) -> CompiledGraph:
    model = model.bind_tools(tools)

    # Define the function that determines which node to go to
    def should_continue(state: ChatAgentState):
        messages = state["messages"]
        last_message = messages[-1]
        # If there are function calls, continue. else, end
        if last_message.get("tool_calls"):
            return "continue"
        else:
            return "end"

    if system_prompt:
        preprocessor = RunnableLambda(
            lambda state: [{"role": "system", "content": system_prompt}]
            + state["messages"]
        )
    else:
        preprocessor = RunnableLambda(lambda state: state["messages"])
    model_runnable = preprocessor | model

    def call_model(
        state: ChatAgentState,
        config: RunnableConfig,
    ):
        response = model_runnable.invoke(state, config)

        return {"messages": [response]}

    workflow = StateGraph(ChatAgentState)

    workflow.add_node("agent", RunnableLambda(call_model))
    workflow.add_node("tools", ChatAgentToolNode(tools))

    workflow.set_entry_point("agent")
    workflow.add_conditional_edges(
        "agent",
        should_continue,
        {
            "continue": "tools",
            "end": END,
        },
    )
    workflow.add_edge("tools", "agent")

    return workflow.compile()


class LangGraphChatAgent(ChatAgent):
    def __init__(self, agent: CompiledStateGraph):
        self.agent = agent

    def predict(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> ChatAgentResponse:
        request = {"messages": self._convert_messages_to_dict(messages)}

        messages = []
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                messages.extend(
                    ChatAgentMessage(**msg) for msg in node_data.get("messages", [])
                )
        return ChatAgentResponse(messages=messages)

    def predict_stream(
        self,
        messages: list[ChatAgentMessage],
        context: Optional[ChatContext] = None,
        custom_inputs: Optional[dict[str, Any]] = None,
    ) -> Generator[ChatAgentChunk, None, None]:
        request = {"messages": self._convert_messages_to_dict(messages)}
        for event in self.agent.stream(request, stream_mode="updates"):
            for node_data in event.values():
                yield from (
                    ChatAgentChunk(**{"delta": msg}) for msg in node_data["messages"]
                )


# Create the agent object, and specify it as the agent object to use when
# loading the agent back for inference via mlflow.models.set_model()
agent = create_tool_calling_agent(llm, tools, system_prompt)
AGENT = LangGraphChatAgent(agent)
mlflow.models.set_model(AGENT)

We now repeat the steps above:
1. Log the model (with a different name) as an MLflow model, and set the new model as the active model
2. Re-run the evaluations and see if we have successfully improved the agent's performance
3. (Optional) continue iterating on the model until we are satisfied with the performance

In [0]:
import mlflow
from agent import LLM_ENDPOINT_NAME, tools
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from pkg_resources import get_distribution
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "I want a refund for order 04e9a339e7fb4435b5a084a60edd927f"
        }
    ]
}

with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent_v2",
        python_model="agent_v2.py",
        input_example=input_example,
        resources=resources,
        pip_requirements=[
            f"databricks-connect=={get_distribution('databricks-connect').version}",
            f"mlflow=={get_distribution('mlflow').version}",
            f"databricks-langchain=={get_distribution('databricks-langchain').version}",
            f"langgraph=={get_distribution('langgraph').version}",
        ],
    )

mlflow.set_active_model(model_id = logged_agent_info.model_id)

Now let's re-run our evaluations.

In [0]:
from agent_v2 import AGENT
import mlflow
from mlflow.genai.scorers import Guidelines

mlflow.set_active_model(name="agent_v2")

refund_queries = [
    # late orders; unrelated reasons
    "My order never arrived! I want a refund. Order f0b46db05b7d4f1dbe29e6878842d00b.",
    "My order was missing hot sauce. Make this right or I am never ordering again! 75d895127eef45e69a43f15b40671926",
    "Please refund my order, my ice cream was melted f883c5d64f4741c182f0567162fc2b06",
    # on-time orders; unrelated reasons
    "My order never arrived! I want a refund. Order 0737517892f447b387fcccd8cbf11c15.",
    "My order was missing hot sauce. Make this right or I am never ordering again! b433aa2fd00446eb8212f86bf20358d1",
    "Please refund my order, my ice cream was melted 48f786e9adbe428e88f97f5aef65a08c",
    # Delayed orders
    "9cffad6a425a4070ac9f6f70ef17761d",
    "My order was really late! 9c63c4433cd44fa0b48149fba5605019",
]

data = []
# Expand data with all the examples in refund_queries
for query in refund_queries:
    data.append(
        {
            "inputs": {
                "messages": [
                    {
                        "role": "user",
                        "content": query,
                    }
                ]
            },
        }
    )


refund_reason = Guidelines(
    name="refund_reason",
    guidelines=["If a refund is offered, its reason must relate to order timing, not to other issues such as missing components."]
)

results = mlflow.genai.evaluate(
    data=data,
    scorers=[refund_reason],
    predict_fn = lambda messages: AGENT.predict({"messages": messages})
)


Now we can use the same approach as above to review the updated evaluation and iterate further if needed.

## Register the model to Unity Catalog

Now that we are satisfied with the performance of our agent, we can register it to Unity Catalog with `mlflow.register_model`.

In [0]:
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = "caspers.default.refund_agent"

# register the model to UC
uc_registered_model_info = mlflow.register_model(
    model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME
)

We can then deploy the model using the `databricks.agents` package. This will allow us to query the model via REST API, the OpenAI SDK, the Playground, and a variety of other means.

In [0]:
from databricks import agents
agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, scale_to_zero=True)

## Query the Deployed Agent

Now you can query the deployed agent right from the playground!

<img src="./images/agents/deployed_agent.png" width="75%"/>

You can also invoke the deployed agent using e.g. Curl or Python. Here's how to invoke the model using the OpenAI SDK:

In [0]:
from openai import OpenAI
from databricks.sdk import WorkspaceClient

# Create a temporary token
w = WorkspaceClient()
tmp_token = w.tokens.create(lifetime_seconds=2400).token_value

client = OpenAI(
    api_key=tmp_token,
    base_url=f"{w.config.host}/serving-endpoints",
)


completion = client.chat.completions.create(
  model="agents_caspers-default-refund_agent",
  messages=[
    {"role": "user", "content": "My order was really late! 9c63c4433cd44fa0b48149fba5605019"}
  ],
)

completion

## Next Steps

In this guide, we prototyped a simple agent using UC tools and the Playground. We then defined the agent in code with LangGraph, evaluated it with MLflow, made some improvements, and deployed it to the Playground.

There's a lot more you can do with agents. For a more detailed guide, check out the Databricks [Build genAI Apps](https://docs.databricks.com/aws/en/generative-ai/agent-framework/build-genai-apps) docs. Some suggestions for next steps:

- Set up [monitoring](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/run-scorer-in-prod) for your deployed agent in order to run scorers on a sample of production traces for continuous monitoring
- Version, manage, and optimize your prompts using the [MLflow prompt registry](https://docs.databricks.com/aws/en/mlflow3/genai/prompt-version-mgmt/prompt-registry/create-and-edit-prompts)
- Collect [human feedback](https://docs.databricks.com/aws/en/mlflow3/genai/human-feedback/) on your agent's responses